In [2]:
import pandas as pd
import numpy as np

In [57]:
ruta_del_archivo = r'C:\Users\Jess\Desktop\parametros areas sin film.xlsx'

df = pd.read_excel(ruta_del_archivo, sheet_name='table')

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
     display(df)


,Solvent,Method,A(010)/A(100)(a.u.),A(100)/A(Side)(a.u.),A(010)/A(Side)(a.u.),A(001)/A(Side)(a.u.),g(100)%oop,g(100)%ip,g(010)%oop,g(010)%ip,g(001)%
0,CB,I,11.394654,0.356902,4.066779,0.148709,17.657473,16.886712,14.169689,11.686793,11.855538
1,CB,I,9.892754,0.326498,3.229968,0.074448,21.469007,15.933777,14.284087,11.728806,12.713575
2,CB,I,7.186339,0.335006,2.407467,0.047170,21.445630,15.635153,14.423789,12.963433,12.975912
3,CB,II,10.915058,0.365303,3.987306,0.045487,21.579432,15.920129,15.109852,11.596123,12.679261
4,CB,II,11.350299,0.253992,2.882890,0.021293,21.183641,16.213129,14.921364,10.398400,12.598649
5,CB,II,7.713026,0.329461,2.541142,0.023556,20.417952,18.794721,14.906922,10.306894,13.245306
6,CB,III,9.717379,0.211849,2.058622,0.070462,21.701677,17.208982,14.515563,14.710945,17.541367
7,CB,III,10.809040,0.286095,3.092415,0.062977,22.095342,17.806898,14.511276,14.809445,11.481520
8,CB,III,11.087424,0.308453,3.419945,0.081498,23.834024,19.199567,13.963080,13.759463,12.828557
9,CB,IV,17.661808,0.237139,4.188307,0.065049,24.464902,18.031002,14.268747,14.187466,13.442903


In [58]:
# 1. Nueva Función: Detección por Coeficiente de Variación (CV > 20%)
def detectar_outlier_cv(values, threshold=0.3):
    """
    Calcula el CV. Si el CV > 20%, asume que hay un artefacto experimental 
    y descarta el valor que esté más alejado de la mediana de la tripleta.
    """
    if len(values) != 3 or any(pd.isna(values)):
        return False, None, 0.0
    
    media = np.mean(values)
    if media == 0: 
        return False, None, 0.0
        
    std = np.std(values, ddof=1) # Desviación estándar muestral
    cv = abs(std / media)
    
    if cv > threshold:
        # Encontrar el valor más alejado de la mediana para descartarlo
        mediana = np.median(values)
        distancias = [abs(v - mediana) for v in values]
        indice_outlier = np.argmax(distancias) # El índice del error más grave
        return True, indice_outlier, cv
        
    return False, None, cv


In [59]:
numeric_cols = ['A(001)', 'A(100)', 'A(010)', 'A(Side)'] 

if not all(col in df.columns for col in numeric_cols):
    numeric_cols = ['A(010)/A(100)(a.u.)', 'A(100)/A(Side)(a.u.)', 'A(010)/A(Side)(a.u.)', 'A(001)/A(Side)(a.u.)']
# Solo analizamos las columnas que realmente existan en el Excel
numeric_cols = [c for c in numeric_cols if c in df.columns]

outliers_posiciones = set()
outliers_info = []

# 3. Detectar las coordenadas de los Outliers con CV > 20%
for name, group in df.groupby(['Solvent', 'Method']):
    if len(group) == 3:
        group_indices = group.index.tolist()
        
        for col in numeric_cols:
            vals = group[col].tolist()
            is_outlier, idx_in_group, cv_val = detectar_outlier_cv(vals, threshold=0.3)
            
            if is_outlier:
                global_idx = group_indices[idx_in_group]
                outliers_posiciones.add((global_idx, col))
                outliers_info.append({
                    'Solvent': name[0],
                    'Method': name[1],
                    'Parametro': col,
                    'Valor_Descartado': vals[idx_in_group],
                    'CV_Calculado': f"{cv_val*100:.1f}%"
                })

In [60]:
# Imprimir un resumen de lo que detectó
print("--- 🚨 REPORTE DE OUTLIERS (Por CV > 20%) ---")
if outliers_info:
    display(pd.DataFrame(outliers_info))
else:
    print("No se encontró ningún parámetro con CV > 20%.\n")

# 4. Función de estilo para colorear de rojo
def resaltar_outliers(data):
    df_colores = pd.DataFrame('background-color: transparent', index=data.index, columns=data.columns)
    for fila, columna in outliers_posiciones:
        df_colores.at[fila, columna] = 'background-color: #ffcccc; color: #cc0000; font-weight: bold;'
    return df_colores

# 5. Aplicar el estilo y mostrar/exportar
tabla_estilizada = df.style.apply(resaltar_outliers, axis=None)

# Guardar el Excel coloreado
tabla_estilizada.to_excel('reporte_outliers_cv_color.xlsx', engine='openpyxl', index=False)

print("\n¡Excel 'reporte_outliers_cv_color.xlsx' generado con éxito!\nAquí tienes la visualización final:")
display(tabla_estilizada)

--- 🚨 REPORTE DE OUTLIERS (Por CV > 20%) ---


,Solvent,Method,Parametro,Valor_Descartado,CV_Calculado
0,CB,I,A(001)/A(Side)(a.u.),0.148709,58.3%
1,CB,II,A(001)/A(Side)(a.u.),0.045487,44.4%
2,CB,IV,A(010)/A(100)(a.u.),6.094150,46.9%
3,CB,IV,A(100)/A(Side)(a.u.),0.478879,38.3%
4,CB,IV,A(001)/A(Side)(a.u.),0.120158,33.4%
5,oXy,I,A(001)/A(Side)(a.u.),0.096040,58.1%
6,oXy,II,A(100)/A(Side)(a.u.),1.752887,97.3%
7,oXy,II,A(010)/A(Side)(a.u.),16.909931,103.3%
8,oXy,II,A(001)/A(Side)(a.u.),0.339492,100.4%



¡Excel 'reporte_outliers_cv_color.xlsx' generado con éxito!
Aquí tienes la visualización final:


,Solvent,Method,A(010)/A(100)(a.u.),A(100)/A(Side)(a.u.),A(010)/A(Side)(a.u.),A(001)/A(Side)(a.u.),g(100)%oop,g(100)%ip,g(010)%oop,g(010)%ip,g(001)%
0,CB,I,11.394654,0.356902,4.066779,0.148709,17.657473,16.886712,14.169689,11.686793,11.855538
1,CB,I,9.892754,0.326498,3.229968,0.074448,21.469007,15.933777,14.284087,11.728806,12.713575
2,CB,I,7.186339,0.335006,2.407467,0.047170,21.445630,15.635153,14.423789,12.963433,12.975912
3,CB,II,10.915058,0.365303,3.987306,0.045487,21.579432,15.920129,15.109852,11.596123,12.679261
4,CB,II,11.350299,0.253992,2.882890,0.021293,21.183641,16.213129,14.921364,10.398400,12.598649
5,CB,II,7.713026,0.329461,2.541142,0.023556,20.417952,18.794721,14.906922,10.306894,13.245306
6,CB,III,9.717379,0.211849,2.058622,0.070462,21.701677,17.208982,14.515563,14.710945,17.541367
7,CB,III,10.809040,0.286095,3.092415,0.062977,22.095342,17.806898,14.511276,14.809445,11.481520
8,CB,III,11.087424,0.308453,3.419945,0.081498,23.834024,19.199567,13.963080,13.759463,12.828557
9,CB,IV,17.661808,0.237139,4.188307,0.065049,24.464902,18.031002,14.268747,14.187466,13.442903
